# Part 1 — Acquisition & Storage
## Notebook 01. FDIC data acquisition and source verification

**Objective.** Retrieve all FDIC Financials observations from 1984Q1 through 2026Q2 and complete Failures and Institutions reference tables. Preserve a typed, quarter-partitioned source snapshot, validate each artifact, and publish a cryptographically identified input for subsequent processing.

**Method.** Public FDIC BankFind Suite API; deterministic field set; paginated requests; bounded retries; atomic Parquet writes; resumable quarter-level progress; SHA-256 verification. Source values, including historical missingness, are preserved without feature construction or outcome labelling.

**Result contract.** A complete `acquisition/` directory with 170 Financials quarter partitions, two reference tables, `manifest.json`, `verification_report.json`, and `acquisition_gate.json`. All downstream notebooks use the selected experiment's signed stage reference.

### 1. Select Google Drive workspace and experiment

A fresh workspace starts a new experiment. An existing workspace resumes its explicitly selected experiment. Set `CREATE_NEW_EXPERIMENT = True` only when you intend to create an independent run; previous experiments are not overwritten. Google Drive is persistent storage; compute-intensive measurements use Colab local disk.

In [ ]:
from google.colab import drive
from pathlib import Path
import json

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/BFT65B_11230945")
CREATE_NEW_EXPERIMENT = False
active_file = DRIVE_ROOT / "control" / "active_experiment.json"
print("Drive root:", DRIVE_ROOT)
print("Existing experiment selected:", active_file.is_file() and not CREATE_NEW_EXPERIMENT)

### 2. Restore the pinned source revision and Python dependencies

The experiment records a complete Git commit SHA. Resumed notebooks check out the original revision, even if the repository receives later notebook-output commits. Running different source revisions within the same experiment is prohibited.

In [ ]:
import shutil
import subprocess
import sys

REPOSITORY = "https://github.com/NguyenDuong58/11230945-NguyenBaDuong.git"
REPO_DIR = Path("/content/bft65b_code")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--quiet", REPOSITORY, str(REPO_DIR)], check=True)
if active_file.exists() and not CREATE_NEW_EXPERIMENT:
    pointer = json.loads(active_file.read_text(encoding="utf-8"))
    cfg_path = DRIVE_ROOT / "experiments" / pointer["experiment_id"] / "experiment_config.lock.json"
    pinned_commit = json.loads(cfg_path.read_text(encoding="utf-8"))["git_commit"]
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--quiet", pinned_commit], check=True)
GIT_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(REPO_DIR / "requirements.txt")], check=True)
sys.path.insert(0, str(REPO_DIR / "src"))
print("Git commit:", GIT_COMMIT)
print("Python:", sys.version.split()[0])

### 3. Execute the acquisition and storage test suite

The test gate must report zero failures, zero errors and zero skipped tests. Integration checks use synthetic FDIC-shaped records; they do not replace the full FDIC acquisition.

In [ ]:
import xml.etree.ElementTree as ET

TEST_XML = Path("/content/bft65b_part1_tests.xml")
subprocess.run([
    sys.executable, "-m", "pytest", "-q", "-rs", "--junitxml", str(TEST_XML),
    str(REPO_DIR / "tests")
], cwd=REPO_DIR, check=True)
suites = ET.parse(TEST_XML).getroot().findall(".//testsuite")
assert suites, "No test results were recorded"
summary = {key: sum(int(s.get(key, 0)) for s in suites)
           for key in ("tests", "failures", "errors", "skipped")}
assert summary["failures"] == summary["errors"] == summary["skipped"] == 0, summary
print("Part 1 test gate: PASSED", summary)

### 4. Initialize the experiment registry

The active experiment ID is shared by both Part 1 notebooks. Its configuration locks the code revision, study interval and fixed seed; source-data artifacts belong to its `acquisition/` stage.

In [ ]:
from fdic_pipeline.project_flow import initialize, get_active, stage_dir, resolve_stage

ACTIVE = initialize(DRIVE_ROOT, GIT_COMMIT, create_new=CREATE_NEW_EXPERIMENT)
EXPERIMENT_ID = ACTIVE["experiment_id"]
SOURCE_DIR = stage_dir(DRIVE_ROOT, "acquisition")
assert ACTIVE["config"]["git_commit"] == GIT_COMMIT
print("Experiment:", EXPERIMENT_ID)
print("Source output:", SOURCE_DIR)

### 5. Acquire the FDIC source tables

Financials are requested in 170 separate calendar-quarter units using offset pagination; Failures and Institutions are requested without date filters. Each completed unit is stored as Snappy Parquet and registered in `manifest.json` only after validation. Restarting an interrupted run reuses valid files and repairs missing or corrupted units. Full extraction can require substantial time; do not interrupt merely because an individual quarter is slow.

In [ ]:
from fdic_pipeline.storage_benchmark import read_json
from fdic_pipeline.project_flow import HandoffError

try:
    existing = resolve_stage(DRIVE_ROOT, "acquisition")
except HandoffError as exc:
    if "has not passed verification" not in str(exc):
        raise
    existing = None

if existing is None:
    ACQ_CMD = [
        sys.executable, str(REPO_DIR / "run_acquisition.py"),
        "--start", "1984-03-31", "--end", "2026-06-30",
        "--out", str(SOURCE_DIR), "--page-size", "10000",
        "--max-retries", "6", "--timeout", "90", "--pause", "0.20",
    ]
    subprocess.run(ACQ_CMD, cwd=REPO_DIR, check=True)
else:
    print("Acquisition stage is already published and immutable; retrieval skipped")

SOURCE_RESULT = read_json(SOURCE_DIR / "verification_report.json")
assert SOURCE_RESULT["success"] is True
assert SOURCE_RESULT["verified_quarters"] == 170
assert not SOURCE_RESULT["missing_quarters"]
assert not SOURCE_RESULT["invalid_artifacts"]
print("Financials:", f'{SOURCE_RESULT["financial_rows"]:,}', "rows")
print("Failures:", f'{SOURCE_RESULT["tables"]["failures"]["rows"]:,}', "rows")
print("Institutions:", f'{SOURCE_RESULT["tables"]["institutions"]["rows"]:,}', "rows")

### 6. Independently verify and publish acquisition provenance

Before handoff, verify the fixed reporting scope and field dictionary, completeness, reference-table availability, and manifest digests. The published gate cannot be silently replaced. No expected row count is copied from a historical experiment.

In [ ]:
from fdic_pipeline.workflow import acquisition_contract
from fdic_pipeline.acquisition import atomic_json, sha256
from fdic_pipeline.project_flow import publish_stage

CHECK = acquisition_contract(SOURCE_DIR)
assert CHECK["quarters"] == 170
assert CHECK["financial_fields"] == 33
assert CHECK["failures_rows"] > 0 and CHECK["institutions_rows"] > 0
GATE_FILE = SOURCE_DIR / "acquisition_gate.json"
if existing is None:
    gate = {
        "success": True, "experiment_id": EXPERIMENT_ID,
        "git_commit": GIT_COMMIT, "source_manifest_sha256": CHECK["manifest_sha256"],
        "source_verification_sha256": CHECK["verification_report_sha256"],
        "financial_rows": CHECK["financial_rows"], "quarters": CHECK["quarters"],
        "distinct_financial_certs": CHECK["distinct_financial_certs"],
        "failures_rows": CHECK["failures_rows"],
        "failures_missing_cert_rows": CHECK["failures_missing_cert_rows"],
        "institutions_rows": CHECK["institutions_rows"],
        "data_interval": ["1984Q1", "2026Q2"],
    }
    atomic_json(GATE_FILE, gate)
    publish_stage(DRIVE_ROOT, "acquisition", "acquisition_gate.json")
else:
    assert read_json(GATE_FILE)["source_manifest_sha256"] == CHECK["manifest_sha256"]
    assert read_json(GATE_FILE)["source_verification_sha256"] == CHECK["verification_report_sha256"]

VERIFIED_STAGE = resolve_stage(DRIVE_ROOT, "acquisition")
assert VERIFIED_STAGE["validation"]["git_commit"] == GIT_COMMIT
print("Acquisition stage: PASSED")
print("Source manifest SHA-256:", CHECK["manifest_sha256"])
print("Verification SHA-256:", CHECK["verification_report_sha256"])
print("Failure records without CERT:", CHECK["failures_missing_cert_rows"])

### 7. Examine source coverage and the storage handoff

Report actual record counts and partition sizes. Missing certificate numbers in historical failure records remain null and must be audited during Part 2; they are not deleted. No failure labels or exit classifications are constructed during acquisition.

In [ ]:
import statistics
MANIFEST = read_json(SOURCE_DIR / "manifest.json")
quarter_rows = [MANIFEST["financials"][q]["rows"] for q in sorted(MANIFEST["financials"])]
quarter_bytes = [MANIFEST["financials"][q]["size_bytes"] for q in sorted(MANIFEST["financials"])]
assert len(quarter_rows) == 170
assert sum(quarter_rows) == CHECK["financial_rows"]
print("Quarter partitions:", len(quarter_rows))
print("Median observations per quarter:", round(statistics.median(quarter_rows)))
print("Partitioned Financials size (MiB):", round(sum(quarter_bytes) / 2**20, 3))
print("Source stage:", VERIFIED_STAGE["path"])
print("Next notebook: notebooks/02_storage_colab.ipynb")